# Clase 4: cargar, explorar y exportar una base de datos
## Instituciones de Educacion Intercultural Bilingue del Ecuador

En esta practica aprenderemos a cargar un archivo CSV en Google Colab, reconocer su estructura, revisar su calidad, responder preguntas sencillas y exportar una version procesada.

**Fuente:** Secretaria de Educacion Intercultural Bilingue y la Etnoeducacion (SEIBE), diciembre de 2022.

**Unidad de observacion:** cada fila representa una institucion educativa.

> Un EDA (analisis exploratorio de datos) sirve para conocer una base antes de realizar analisis mas complejos. No busca probar una hipotesis ni establecer causalidad.

## 1. ¿Donde puede estar alojada la base?

Colab se ejecuta en una computadora temporal. Por eso, un archivo subido directamente desaparece cuando termina la sesion. Tenemos tres opciones:

1. **En una pagina web:** Colab lee el CSV mediante su URL. Es la opcion principal de este cuaderno.
2. **En nuestra computadora:** subimos el archivo durante la clase. Es sencillo, pero temporal.
3. **En Google Drive:** el archivo permanece guardado y puede reutilizarse.

Ejecutaremos solo una de las tres opciones.

## 2. Opcion principal: cargar desde la fuente oficial

La direccion apunta directamente al archivo CSV oficial. Esta base usa `;` para separar las columnas y contiene una marca de codificacion UTF-8; por eso utilizamos `sep=';'` y `encoding='utf-8-sig'`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

url = 'https://www.datosabiertos.gob.ec/dataset/681f9ed6-0536-44b8-b771-903506c822a1/resource/65d8aa62-9483-4fe9-b8ff-ec5772406aca/download/seibe_institucioneducativa_2022dic.csv'
df = pd.read_csv(url, sep=';', encoding='utf-8-sig')

print('Base cargada correctamente')

### Alternativa A: subir el archivo desde la computadora

Usa esta opcion si el enlace no funciona. Primero descarga el CSV y despues ejecuta la celda. Colab mostrara un boton para seleccionar el archivo. **No ejecutes esta celda si ya cargaste la base desde la URL.**

In [ ]:
# Quita el simbolo # de las lineas siguientes si quieres usar esta opcion.
# from google.colab import files
# archivos = files.upload()
# nombre_archivo = next(iter(archivos))
# df = pd.read_csv(nombre_archivo, sep=';', encoding='utf-8-sig')

### Alternativa B: leer el archivo desde Google Drive

Guarda el CSV, por ejemplo, en `Mi unidad/FLACSO/datos/`. La primera vez, Google pedira permiso para conectar Drive. **No ejecutes esta celda si ya cargaste la base con otra opcion.**

In [ ]:
# Quita el simbolo # de las lineas siguientes y ajusta la ruta.
# from google.colab import drive
# drive.mount('/content/drive')
# ruta = '/content/drive/MyDrive/FLACSO/datos/SEIBE_InstitucionEducativa_2022Dic.csv'
# df = pd.read_csv(ruta, sep=';', encoding='utf-8-sig')

## 3. Primera mirada a la base

`head()` muestra las primeras cinco filas. Nos permite comprobar si las columnas se separaron correctamente y entender que representa cada registro.

In [ ]:
df.head()

`shape` devuelve dos numeros: cantidad de filas y cantidad de columnas. `columns` muestra los nombres de las variables.

In [ ]:
print(f'Filas: {df.shape[0]}')
print(f'Columnas: {df.shape[1]}')
print('\nNombres de las columnas:')
print(df.columns.tolist())

## 4. Tipos de variables

`info()` muestra el tipo de cada columna y cuantos valores no nulos contiene. En pandas, `object` suele indicar texto; `int64` y `float64` indican numeros.

In [ ]:
df.info()

## 5. Calidad de los datos

Revisaremos valores faltantes y filas duplicadas. Un valor faltante no siempre es un error: primero debemos entender por que falta y que significa.

In [ ]:
faltantes = df.isna().sum().sort_values(ascending=False)
faltantes = faltantes[faltantes > 0]
print('Valores faltantes por columna:')
display(faltantes)
print(f'Filas completamente duplicadas: {df.duplicated().sum()}')

In [ ]:
# Observamos las filas que tienen datos faltantes.
df[df.isna().any(axis=1)]

## 6. Estadisticas descriptivas

`describe()` resume las variables numericas: cantidad de observaciones, promedio, desviacion estandar, minimo, cuartiles y maximo. Un promedio por si solo no describe toda la distribucion.

In [ ]:
df[['Número Alumnos', 'Número Docentes']].describe().round(2)

Para las variables categoricas podemos contar cuantas veces aparece cada categoria con `value_counts()`.

In [ ]:
print('Instituciones por sostenimiento:')
display(df['Sostenimiento'].value_counts())

print('Instituciones por regimen escolar:')
display(df['Régimen Escolar'].value_counts())

## 7. Agrupar para responder preguntas

Pregunta 1: ¿que provincias tienen mas instituciones registradas? `groupby()` forma grupos y `size()` cuenta las filas de cada grupo.

In [ ]:
instituciones_provincia = (
    df.groupby('Provincia')
      .size()
      .sort_values(ascending=False)
      .reset_index(name='Número de instituciones')
)
instituciones_provincia.head(10)

Pregunta 2: ¿que provincias registran mas estudiantes en estas instituciones?

In [ ]:
estudiantes_provincia = (
    df.groupby('Provincia', as_index=False)['Número Alumnos']
      .sum()
      .sort_values('Número Alumnos', ascending=False)
)
estudiantes_provincia.head(10)

## 8. Visualizaciones basicas

Un grafico de barras facilita comparar categorias. Mostraremos las diez provincias con mas instituciones.

In [ ]:
top10 = instituciones_provincia.head(10)

plt.figure(figsize=(10, 6))
sns.barplot(data=top10, x='Número de instituciones', y='Provincia', color='#6955A5')
plt.title('Diez provincias con mas instituciones registradas')
plt.xlabel('Numero de instituciones')
plt.ylabel('Provincia')
plt.tight_layout()
plt.show()

Un histograma muestra como se distribuye una variable numerica. Limitamos el eje a 500 estudiantes para observar mejor la mayoria de instituciones; esto no elimina datos ni significa que no existan valores mayores.

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(data=df, x='Número Alumnos', bins=30, color='#2A9D8F')
plt.xlim(0, 500)
plt.title('Distribucion del numero de estudiantes por institucion')
plt.xlabel('Numero de estudiantes')
plt.ylabel('Cantidad de instituciones')
plt.tight_layout()
plt.show()

## 9. Crear una nueva variable

Calcularemos estudiantes por docente. Es una razon descriptiva construida a partir de dos columnas. No debe interpretarse automaticamente como calidad educativa, carga laboral ni tamano exacto de aula.

In [ ]:
df['Estudiantes por docente'] = (df['Número Alumnos'] / df['Número Docentes']).round(2)
df[['Nombre Institución', 'Número Alumnos', 'Número Docentes', 'Estudiantes por docente']].head()

## 10. Exportar la base procesada

Exportaremos el DataFrame como un nuevo CSV. `index=False` evita crear una columna adicional con el indice de pandas. `encoding='utf-8-sig'` ayuda a conservar tildes y enes al abrir el archivo en Excel.

In [ ]:
nombre_salida = 'SEIBE_instituciones_procesada.csv'
df.to_csv(nombre_salida, index=False, encoding='utf-8-sig')
print(f'Archivo creado: {nombre_salida}')

In [ ]:
# En Google Colab, esta celda descarga el archivo a la computadora.
from google.colab import files
files.download(nombre_salida)

## 11. Preguntas para interpretar los resultados

1. ¿Que variables son numericas y cuales son categoricas?
2. ¿Que provincia concentra mas instituciones? ¿Eso significa que tiene mejor cobertura?
3. ¿Por que el promedio y la mediana de estudiantes pueden ser diferentes?
4. ¿Que decisiones tomariamos frente a los valores faltantes?
5. ¿Que informacion necesitariamos para analizar acceso o calidad educativa?
6. ¿Que categorias utiliza la base para representar identidades etnicas, nacionalidades y pueblos? ¿Quien definio esas categorias?

> La base describe las instituciones incluidas en este registro en diciembre de 2022. No permite, por si sola, explicar las causas de las diferencias territoriales ni evaluar la calidad educativa.